In [ ]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import sympy.physics.mechanics as me
import sys 
sys.path.insert(0, "..")
from importlib import reload
import matplotlib.pyplot as plt
import equations as eq
reload (eq);
import trajectory_lib as tr
reload (tr);

participant = 'par2'
GH_seq = 'YZY'

# load OS struct created by readosim.m funcion
OS_struct = sc.io.loadmat('../Motions/'+participant+'/OS_model_prediction.mat')

# build equations of motion
# fr + frstar = 0 is implicitly defined equations of motion formed by Kane's method, 
# q, u, faux are the generalized coordinates, speeds and GH reaction forces, respectively.
q,u,faux,fr,frstar,kinematical = eq.create_eoms_quat_w_RF(OS_struct,weight = 0,derive = 'numeric',gen_matlab_functions = 0)

[0.16593663903022232, 0.7503314434097361, 2.1825986514494433, 0.5812037151673941, 0.5812037151673941, 0.5584406121209405]


In [13]:
# coordinates and speeds
print(q)
print(u)

[q0_clavicula(t), q1_clavicula(t), q2_clavicula(t), q3_clavicula(t), q0_scapula(t), q1_scapula(t), q2_scapula(t), q3_scapula(t), q0_humerus(t), q1_humerus(t), q2_humerus(t), q3_humerus(t), q_ulna(t)]
[w1_clavicula(t), w2_clavicula(t), w3_clavicula(t), w1_scapula(t), w2_scapula(t), w3_scapula(t), w1_humerus(t), w2_humerus(t), w3_humerus(t), w_ulna(t)]


In [14]:
# first 10 elements of fr+frstar are the equations of motion
# the last 3 elements are implicitly defined GH reaction forces from dynamics
# do not print the full fr+frstar because it is very long and would crash the notebook
len(fr+frstar)

13

In [15]:
reload(eq)
clav_pos = 0.4
tilt_y = 13
tilt_z = -6.5
w_traj = 200
w_optim_params = 1
simulation = 'All_motions'
w_optim_fmax = 5
w_optim_lceopt = 5
isim = 0
EMG_weight = 2
wGH = 2
w_act = 1
w_diff_vel = 1e-2
w_diff_exc = 1e-3
w_diff_faux = 1e-2
include_activation_dynamics = True
thor_hum_only = False

# TE is the polynomial-based spatial torque from muscle forces,
# activations is the list of symbolic muscle activations,
# TE_conoid is the spatial torque from conoid ligament
# 
TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,calibrated_params = 'calibrate_params', derive = 'numeric', RC_lim = 1.0)
params_init = {**fmax_init, **lceopt_init}
params_range = {**fmax_range, **lceopt_range}
myKeys = list(params_init.keys())
myKeys.sort()
sd = {i: params_init[i] for i in myKeys}
sorted_params_init = np.array([*sd.values()])
num_params = len(sorted_params_init)


In [16]:
# one group muscle scaler scales the whole muscle group
print(mus_groups)

[['deltscap1', 'deltscap2', 'deltscap3', 'deltscap4', 'deltscap5', 'deltscap6', 'deltscap7', 'deltscap8', 'deltscap9', 'deltscap10', 'deltscap11'], ['deltclav1', 'deltclav2', 'deltclav3', 'deltclav4'], ['trapscap1', 'trapscap2', 'trapscap3', 'trapscap4', 'trapscap5', 'trapscap6', 'trapscap7', 'trapscap8', 'trapscap9', 'trapscap10', 'trapscap11'], ['trapclav1', 'trapclav2'], ['serrant1', 'serrant2', 'serrant3', 'serrant4', 'serrant5', 'serrant6', 'serrant7', 'serrant8', 'serrant9', 'serrant10', 'serrant11'], ['infra1', 'infra2', 'infra3', 'infra4', 'infra5', 'infra6']]


In [17]:
# EMG names in the EMG struct and the activations that corresponds to those EMG signals in the same order.
# i.e. act_47 and act_48 are delt_clav_1 and delt_clav_2, which corresponds to the EMG signal AnteriorDelt in the EMG struct.
emg_names = ('AnteriorDelt','IntermediateDelt','PosteriorDelt','Infrasp','Suprasp','MiddleTrap','UpperTrap','Serrupper')
muscles_emg_tracked = [['act_47','act_48'],['act_43','act_44','act_45','act_46'],['act_38','act_39','act_40'],['act_56','act_57','act_58'],['act_1','act_2'],['act_6','act_7','act_8'],['act_12'],['act_26','act_27','act_28']]

In [ ]:
# add muscle forces and conoid ligament forces to equations of motion, add kinematical differential equations and add GH muscle forces to the reaction forces
eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+sp.Matrix([TE+sp.Matrix(TE_conoid)]).col_join(GH_mus_forces))

# if activation dynamics is included, add activation dynamics equations to the system of equations, and add excitations as specified variables
if include_activation_dynamics:
    excitations = []
    act_ode = []
    for i in range(len(activations)):
        excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
        current_mus_ind = int(str(activations[i])[4:-3])
        current_mus = OS_struct['model']['muscles'].item()[0,(current_mus_ind-1)]
        t_act = current_mus['tact'][0,0].item()
        t_deact = current_mus['tdeact'][0,0].item()
        act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i],t_act,t_deact))
    sp_act_ode = sp.Matrix(act_ode)
    eoms_implicit = eoms_implicit.col_join(sp_act_ode)

interval_value = 0.04
file = '../Motions/' + participant + '/' + simulation + '/' + simulation
traj_original,omega, num_nodes, time = tr.exp_trajectory_quat(file,interval_value)
q0_t0 = traj_original[:,0][:4]
traj = tr.exp_trajectory_quat_myobj(traj_original,clav_pos)
emg, indexes_emg = tr.exp_emg('../Motions/'+participant+'/'+simulation+'/EMG_'+participant+'_'+simulation+'.mat', num_nodes = num_nodes, emg_names = emg_names)

# track clavicle and scapula in all nodes
index_clav_scap = 1
# track humerus only in the nodes where EMG data is defined, len(indexes_emg) = num_nodes
index_hum = indexes_emg

if include_activation_dynamics:
    state_symbols = tuple(q+u+faux+activations)
    specified_symbols = tuple(excitations)
else:
    state_symbols = tuple(q+u+faux)
    specified_symbols = tuple(activations)

num_states = len(state_symbols) 
num_q = len(q)
num_u = len(u)
num_faux = len(faux)
num_inputs = len(specified_symbols)
t = me.dynamicsymbols._t
reload(eq)
# create objective functions and their jacobians
# trajectory tracking and clavicular orientation at t0
objective_traj,objective_traj_jac, objective_SC_t0, objective_SC_t0_jac = eq.objective_traj_quat(num_q,interval_value,clav_pos,True)

objective_act,objective_act_jac = eq.objective_min_activation(activations,interval_value,True,muscles_emg_tracked)

objective_exc,objective_exc_jac = eq.objective_min_activation(activations,interval_value,True,muscles_emg_tracked)

objective_maxstab, objective_maxstab_jac = eq.objective_max_GH_stab(tilt_y=tilt_y,tilt_z=tilt_z,interval_value = interval_value)

obj_min_diff,obj_min_diff_jac = eq.objective_state_diff(num_nodes,interval_value)

obj_cal_params, obj_cal_params_jac = eq.objective_scalers_restriction(mus_groups,w_optim_fmax,w_optim_lceopt)

5*(1 - fmax_scaler_deltclav)**2 + 5*(1 - fmax_scaler_deltscap)**2 + 5*(1 - fmax_scaler_infra)**2 + 5*(1 - fmax_scaler_serrant)**2 + 5*(1 - fmax_scaler_trapclav)**2 + 5*(1 - fmax_scaler_trapscap)**2 + 5*(1 - lceopt_scaler_deltclav)**2 + 5*(1 - lceopt_scaler_deltscap)**2 + 5*(1 - lceopt_scaler_infra)**2 + 5*(1 - lceopt_scaler_serrant)**2 + 5*(1 - lceopt_scaler_trapclav)**2 + 5*(1 - lceopt_scaler_trapscap)**2


In [ ]:
# indexes are defined because of the way the objective function is defined
# we minimize activation squared, we do not minimize excitation
w_min_squared_act = 1
w_min_squared_exc = 0

if include_activation_dynamics:
    w_min_emg_act = 0
    w_min_emg_exc = EMG_weight
    # if activation dynamics is included, match EMG with corresponding excitations
else:
    # if we do not have activation dynamics, we match activations directly
    w_min_emg_act = EMG_weight
    w_min_emg_exc = 0


def obj(free):
    min_traj = w_traj * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj,index_clav_scap,index_hum))
    min_SC_t0 = w_traj * np.sum(objective_SC_t0(free[0::num_nodes][:4],q0_t0))

    min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
    min_faux_dif = w_diff_faux * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))))

    min_act = w_act * np.sum(objective_act(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs),emg,indexes_emg,w_min_squared_act,w_min_emg_act))

    min_instab = wGH * np.sum(objective_maxstab(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

    cal_params = w_optim_params * obj_cal_params(*free[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes])

    obj = (min_traj + min_vel_dif + min_act + cal_params + min_instab + min_faux_dif + min_SC_t0) #   

    if include_activation_dynamics:
        min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))))
        min_exc = w_act * np.sum(objective_exc(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs),emg,indexes_emg,w_min_squared_exc,w_min_emg_exc))
        obj += (min_exc_dif + min_exc)

    return obj.item()

def obj_grad(free):
    grad = np.zeros_like(free)

    grad[:num_q*num_nodes] += w_traj * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj,index_clav_scap,index_hum))
    grad[0::num_nodes][:4] += w_traj * np.sum(objective_SC_t0_jac(free[0::num_nodes][:4],q0_t0))

    grad[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes] += w_act * np.concatenate(objective_act_jac(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs),emg,indexes_emg,w_min_squared_act,w_min_emg_act))
    grad[num_q*num_nodes:(num_q + num_u)*num_nodes] += w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))
    grad[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes] += w_diff_faux * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))[0,:,:])))

    grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes] += wGH * np.concatenate(objective_maxstab_jac(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

    grad[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes] += w_optim_params * obj_cal_params_jac(*free[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes])[0]

    if include_activation_dynamics:
        grad[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes] += w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))[0,:,:])))
        grad[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes] += w_act * np.concatenate(objective_exc_jac(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs),emg,indexes_emg,w_min_squared_exc,w_min_emg_exc))

    return grad

# check objective and gradient values at a random point
print('obj_check', obj(np.ones(num_states*num_nodes + num_inputs*num_nodes + num_params)*0.01))
print('obj_grad_check', sum(obj_grad(np.ones(num_states*num_nodes + num_inputs*num_nodes + num_params)*0.01)))

instance_constraints = []
# generalized speeds are 0 at the first and last node
instance_constraints.append(state_symbols[13].func(time[-1]))
instance_constraints.append(state_symbols[14].func(time[-1]))
instance_constraints.append(state_symbols[15].func(time[-1]))
instance_constraints.append(state_symbols[16].func(time[-1]))
instance_constraints.append(state_symbols[17].func(time[-1]))
instance_constraints.append(state_symbols[18].func(time[-1]))
instance_constraints.append(state_symbols[19].func(time[-1]))
instance_constraints.append(state_symbols[20].func(time[-1]))
instance_constraints.append(state_symbols[21].func(time[-1]))
instance_constraints.append(state_symbols[13].func(time[0]))
instance_constraints.append(state_symbols[14].func(time[0]))
instance_constraints.append(state_symbols[15].func(time[0]))
instance_constraints.append(state_symbols[16].func(time[0]))
instance_constraints.append(state_symbols[17].func(time[0]))
instance_constraints.append(state_symbols[18].func(time[0]))
instance_constraints.append(state_symbols[19].func(time[0]))
instance_constraints.append(state_symbols[20].func(time[0]))
instance_constraints.append(state_symbols[21].func(time[0]))

# unit norm of quaternions at the first node
instance_constraints.append(state_symbols[0].func(0)**2 + state_symbols[1].func(0)**2 + state_symbols[2].func(0)**2 + state_symbols[3].func(0)**2 - 1) # SC
instance_constraints.append(state_symbols[4].func(0)**2 + state_symbols[5].func(0)**2 + state_symbols[6].func(0)**2 + state_symbols[7].func(0)**2 - 1) # AC
instance_constraints.append(state_symbols[8].func(0)**2 + state_symbols[9].func(0)**2 + state_symbols[10].func(0)**2 + state_symbols[11].func(0)**2 - 1) # GH

# bounds for activations and excitations
bounds_act = (0.0,1.0)
bounds = (bounds_act,)*len(activations)
bndrs = dict(zip(activations,bounds))
if include_activation_dynamics:
    bndrs_exc = dict(zip(excitations,(bounds_act,)*len(excitations)))
    bndrs.update(bndrs_exc)

# bounds for states and inputs, we are tracking fully, so we can set tight bounds
for i in range(num_q):
    bndrs.update({q[i]: (min(traj_original[i,:])-0.05, max(traj_original[i,:])+0.05)})

bndrs.update({faux[0]: (-2,0)})
bndrs.update(params_range)


start = tm.time()
prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
            num_nodes, interval_value,
            known_parameter_map={},
            instance_constraints=instance_constraints,
            bounds=bndrs,
            integration_method='midpoint',
            parallel = False)


time_to_create = tm.time() - start
print(time_to_create)

prob.add_option('limited_memory_max_history', 40)
initial_guess = np.ones(prob.num_free)*0.0
time_2_solve_start = tm.time()

prob.add_option('max_iter',2000)
initial_guess[:13*num_nodes] = traj_original.flatten()
initial_guess[(num_q + num_u)*num_nodes:(num_q + num_u + 1)*num_nodes] = -0.75
initial_guess[num_q * num_nodes : (num_q + num_u) * num_nodes] = omega.flatten()

initial_guess[(num_states+num_inputs)*num_nodes:(num_states+num_inputs+num_params)*num_nodes] = np.ones(num_params)
    
solution, info = prob.solve(initial_guess)
time_2_solve = tm.time() - time_2_solve_start
print(info['status_msg'])
print(info['obj_val'])
act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
objective_value = prob.obj_value
print('Objective activations: ', act_obj)

# save result to matlab struct
file_name = '../Motions/'+participant+'/'+simulation+'/' + simulation + '_params_calibration.mat'
tr.sol2struct(solution,activations,num_q,num_u,num_faux,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

# save calibrated parameters to a separate mat file
calibrated_params = dict(zip(myKeys, solution[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes]))
sc.io.savemat('../Motions/'+participant+'/calibrated_params.mat', {'calibrated_params': calibrated_params})